In [ ]:
# -*- coding: utf-8 -*-
"""
SFT数据质量评估与清洗模块（混合评分版）
- 支持真实数据集（JSON/JSONL），自动适配字段
- 多维度评分：完整性、指令遵循、流畅性、安全性
- 评分策略：规则评分(40%) + 大模型评分(60%)，支持自适应采样与回退
- 精确去重 + 近似去重（SimHash）
- 低质量样本聚类分析 + HTML报告
"""

import pandas as pd
import numpy as np
import re
import hashlib
import json
import logging
import os
import glob
import random
from typing import Optional, Tuple, Dict, Any, List
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from dataclasses import dataclass, field
import matplotlib.pyplot as plt

# 可选依赖：如需使用LLM评分，请安装 openai
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)


# ==================== SimHash（近似去重）====================
class Simhash:
    def __init__(self, text, f=64):
        self.f = f
        self.hash = self._compute(text) if text else 0

    def _compute(self, text):
        v = [0] * self.f
        for i in range(len(text)-1):
            gram = text[i:i+2]
            h = hash(gram) & ((1 << self.f) - 1)
            for j in range(self.f):
                if (h >> j) & 1:
                    v[j] += 1
                else:
                    v[j] -= 1
        fingerprint = 0
        for j in range(self.f):
            if v[j] > 0:
                fingerprint |= (1 << j)
        return fingerprint

    def distance(self, other):
        x = self.hash ^ other.hash
        return bin(x).count('1')


@dataclass
class Config:
    """配置类"""
    # 数据
    n_samples: Optional[int] = None          # 采样数量，None表示全部
    random_seed: int = 42
    # 去重
    exact_dedup: bool = True
    near_dedup: bool = True
    simhash_threshold: float = 0.95          # 汉明距离 <= 3 视为重复
    # 评分与筛选
    low_score_threshold: int = 6
    high_score_threshold: int = 8
    # 聚类
    n_clusters: int = 3
    tfidf_max_features: int = 100
    # 输出
    output_dir: str = "output_sft"
    high_quality_file: str = "high_quality_sft.json"
    low_quality_file: str = "low_quality_sft.csv"
    cluster_report_file: str = "cluster_report.html"
    # 优化选项：对未知任务类型启用通用相关性评分
    enable_generic_following: bool = True

    # ========== 新增：LLM混合评分配置 ==========
    enable_llm_scoring: bool = True          # 是否启用LLM评分
    llm_api_key: Optional[str] = "sk-6967a27611be4be0b22bad76d8e45e12"       # API Key，默认从环境变量 DEEPSEEK_API_KEY 读取
    llm_base_url: str = "https://api.deepseek.com"
    llm_model: str = "deepseek-chat"
    llm_sample_ratio: float = 0.1             # 仅对10%的样本启用LLM评分（控制成本）
    llm_score_lower: int = 5                  # 仅对规则总分 >= 此值的样本考虑启用LLM
    llm_score_upper: int = 8                  # 仅对规则总分 <= 此值的样本考虑启用LLM
    llm_rule_weight: float = 0.4              # 规则评分权重
    llm_model_weight: float = 0.6             # 大模型评分权重


class QualityScorer:
    """多维度评分器（规则+大模型混合）"""
    def __init__(self, config: Config):
        self.config = config
        self.llm_client = None

    def _init_llm_client(self):
        """懒加载LLM客户端"""
        if self.llm_client is None and OpenAI is not None:
            api_key = self.config.llm_api_key or os.environ.get("DEEPSEEK_API_KEY")
            if not api_key:
                logger.warning("未找到 LLM API Key，将回退至纯规则评分")
                return
            self.llm_client = OpenAI(
                api_key=api_key,
                base_url=self.config.llm_base_url
            )
            logger.info("LLM 客户端初始化成功")

    def _rule_score(self, instruction: str, output: str, task_type: str) -> Tuple[int, int, int, int, int]:
        """纯规则评分（原有逻辑）"""
        out_len = len(output)

        # 1. 完整性 (0-3)
        if out_len == 0:
            completeness = 0
        elif out_len < 10:
            completeness = 1
        elif out_len < 30:
            completeness = 2
        else:
            completeness = 3

        # 2. 指令遵循 (0-3)
        following = 1
        inst_keywords = set(re.findall(r'[\u4e00-\u9fff]{2,}', instruction))

        if "代码" in task_type:
            if "```" in output or "def " in output:
                following += 2
            elif "return" in output or "print" in output:
                following += 1
            if "while True" in output and "break" not in output:
                following -= 1
        elif "翻译" in task_type:
            if any('\u4e00' <= c <= '\u9fff' for c in output):
                following += 2
            if out_len < len(instruction) * 0.3:
                following -= 1
        elif "问答" in task_type:
            out_keywords = set(re.findall(r'[\u4e00-\u9fff]{2,}', output))
            overlap = len(inst_keywords & out_keywords) / max(1, len(inst_keywords))
            if overlap > 0.3:
                following += 1
            if overlap > 0.6:
                following += 1
        elif "推理" in task_type:
            logic_words = ["因为", "所以", "因此", "如果", "那么", "则"]
            if any(w in output for w in logic_words):
                following += 2
        elif "创意" in task_type:
            if len(set(output)) / max(1, len(output)) < 0.5:
                following -= 2

        if out_len > 20 and inst_keywords:
            out_keywords = set(re.findall(r'[\u4e00-\u9fff]{2,}', output))
            if not (inst_keywords & out_keywords):
                following -= 1

        if any(phrase in output for phrase in ["我不知道", "无法回答", "超出范围"]):
            following = max(0, following - 2)

        if self.config.enable_generic_following and task_type == "未知" and out_len > 20 and inst_keywords:
            out_keywords = set(re.findall(r'[\u4e00-\u9fff]{2,}', output))
            overlap = len(inst_keywords & out_keywords) / max(1, len(inst_keywords))
            if overlap > 0.2:
                following += 1
            if overlap > 0.5:
                following += 1
            following = min(3, following)

        following = min(3, max(0, following))

        # 3. 流畅性 (0-2)
        fluency = 2
        if re.search(r'(.)\1{4,}', output):
            fluency -= 1
        garbage = re.findall(r'[^a-zA-Z\u4e00-\u9fff\s，。！？；：""''《》【】（）]', output)
        if len(garbage) > out_len * 0.3:
            fluency -= 1
        if not re.search(r'[\u4e00-\u9fff]', output) and out_len > 100:
            fluency -= 1
        fluency = max(0, fluency)

        # 4. 安全性 (0-2)
        safety = 2
        if out_len < 5:
            safety = 0
        harmful = ["暴力", "色情", "非法", "诈骗"]
        if any(w in output for w in harmful):
            safety = max(0, safety - 1)

        total = completeness + following + fluency + safety
        return total, completeness, following, fluency, safety

    def _llm_score(self, instruction: str, output: str) -> Optional[Tuple[int, int, int]]:
        """
        调用大模型评分
        返回: (following_score, fluency_score, safety_score) 或 None（失败时）
        """
        if self.llm_client is None:
            return None

        prompt = f"""请对以下指令-输出对进行质量评分，仅返回 JSON 格式。

评分标准：
- instruction_following (0-3分)：输出是否准确、完整地遵循了指令？
- fluency (0-2分)：语言是否流畅自然，无重复、乱码？
- safety (0-2分)：内容是否安全无害？

指令：{instruction[:500]}
输出：{output[:500]}

请返回 JSON：{{"instruction_following": 整数, "fluency": 整数, "safety": 整数}}"""

        try:
            response = self.llm_client.chat.completions.create(
                model=self.config.llm_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=100
            )
            content = response.choices[0].message.content.strip()
            # 尝试提取JSON
            json_match = re.search(r'\{.*\}', content, re.DOTALL)
            if json_match:
                result = json.loads(json_match.group())
            else:
                result = json.loads(content)
            follow = min(3, max(0, int(result.get("instruction_following", 2))))
            flu = min(2, max(0, int(result.get("fluency", 2))))
            safe = min(2, max(0, int(result.get("safety", 2))))
            return follow, flu, safe
        except Exception as e:
            logger.debug(f"LLM评分失败: {e}")
            return None

    def score(self, instruction: Any, input_text: Any, output: Any, task_type: str = "") -> Tuple[int, int, int, int, int]:
        """
        混合评分：规则评分(40%) + 大模型评分(60%)
        - 完整性始终由规则判定
        - 是否启用LLM取决于配置及采样/边界条件
        """
        inst = str(instruction) if instruction is not None else ""
        out = str(output) if output is not None else ""
        task_type = task_type if task_type is not None else ""

        # 1. 获取规则评分
        rule_total, rule_comp, rule_follow, rule_flu, rule_safe = self._rule_score(inst, out, task_type)

        # 2. 决定是否使用LLM评分
        use_llm = False
        if self.config.enable_llm_scoring:
            # 边界条件：规则总分在 [lower, upper] 区间内
            if self.config.llm_score_lower <= rule_total <= self.config.llm_score_upper:
                # 采样比例控制
                if random.random() < self.config.llm_sample_ratio:
                    use_llm = True
                    self._init_llm_client()

        if use_llm:
            llm_result = self._llm_score(inst, out)
            if llm_result is not None:
                llm_follow, llm_flu, llm_safe = llm_result
                # 混合计算
                w_rule = self.config.llm_rule_weight
                w_llm = self.config.llm_model_weight
                final_follow = int(round(w_rule * rule_follow + w_llm * llm_follow))
                final_flu = int(round(w_rule * rule_flu + w_llm * llm_flu))
                final_safe = int(round(w_rule * rule_safe + w_llm * llm_safe))
                final_total = rule_comp + final_follow + final_flu + final_safe
                return final_total, rule_comp, final_follow, final_flu, final_safe
            else:
                logger.debug("LLM评分失败，回退至规则评分")

        # 默认返回规则评分
        return rule_total, rule_comp, rule_follow, rule_flu, rule_safe

    def evaluate_dataset(self, df: pd.DataFrame) -> pd.DataFrame:
        logger.info("开始批量质量评估...")
        if self.config.enable_llm_scoring:
            logger.info(f"LLM混合评分已启用 (采样比例={self.config.llm_sample_ratio:.0%}, 边界=[{self.config.llm_score_lower}, {self.config.llm_score_upper}])")
        scores = []
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="评估进度"):
            total, comp, follow, flu, safe = self.score(
                row['instruction'], row.get('input', ''), row['output'], row.get('task_type', '')
            )
            scores.append({
                'total_score': total,
                'completeness': comp,
                'instruction_following': follow,
                'fluency': flu,
                'safety': safe
            })
        score_df = pd.DataFrame(scores)
        result = pd.concat([df, score_df], axis=1)
        logger.info("评估完成")
        return result


class Deduplicator:
    def __init__(self, config: Config):
        self.config = config

    def exact_deduplication(self, df: pd.DataFrame) -> pd.DataFrame:
        before = len(df)
        df['hash'] = df.apply(lambda row: hashlib.md5(
            (str(row['instruction']) + str(row['output'])).encode('utf-8')
        ).hexdigest(), axis=1)
        df = df.drop_duplicates(subset=['hash']).reset_index(drop=True)
        after = len(df)
        logger.info(f"精确去重: {before} -> {after} (去重率 {(1-after/before)*100:.2f}%)")
        df.drop(columns=['hash'], inplace=True)
        return df

    def near_deduplication(self, df: pd.DataFrame) -> pd.DataFrame:
        if not self.config.near_dedup:
            return df
        def simhash_func(row):
            text = str(row['instruction']) + " [SEP] " + str(row['output'])
            return Simhash(text) if len(text) > 0 else None

        df['simhash'] = df.apply(simhash_func, axis=1)
        keep = []
        hash_list = []
        max_distance = int(64 * (1 - self.config.simhash_threshold))
        for idx, row in df.iterrows():
            h = row['simhash']
            if h is None:
                keep.append(idx)
                continue
            duplicate = False
            for existing in hash_list:
                if h.distance(existing) <= max_distance:
                    duplicate = True
                    break
            if not duplicate:
                hash_list.append(h)
                keep.append(idx)
        before = len(df)
        df = df.iloc[keep].reset_index(drop=True)
        after = len(df)
        logger.info(f"近似去重: {before} -> {after} (去重率 {(1-after/before)*100:.2f}%)")
        df.drop(columns=['simhash'], inplace=True)
        return df


class BadcaseClusterer:
    def __init__(self, config: Config):
        self.config = config

    def cluster(self, df: pd.DataFrame, score_threshold: int = None, verbose: bool = False) -> Optional[pd.DataFrame]:
        if score_threshold is None:
            score_threshold = self.config.low_score_threshold
        low = df[df['total_score'] < score_threshold].copy()
        if len(low) < 3:
            if verbose:
                logger.info(f"低质量样本不足3条（实际{len(low)}），跳过聚类")
            return None
        low['instruction'] = low['instruction'].fillna('').astype(str)
        low = low[low['instruction'].str.strip() != '']
        if len(low) < 3:
            if verbose:
                logger.info("有效指令不足3条，跳过聚类")
            return None
        texts = low['instruction'].tolist()
        try:
            vectorizer = TfidfVectorizer(max_features=self.config.tfidf_max_features)
            X = vectorizer.fit_transform(texts)
            n_clusters = min(self.config.n_clusters, len(texts))
            kmeans = KMeans(n_clusters=n_clusters, random_state=self.config.random_seed, n_init=10)
            labels = kmeans.fit_predict(X)
            low['cluster'] = labels
            if verbose:
                logger.info(f"聚类完成，共 {n_clusters} 类")
            return low
        except Exception as e:
            logger.error(f"聚类失败: {e}")
            return None

    def generate_cluster_report(self, df_clustered: pd.DataFrame, output_path: str):
        if df_clustered is None:
            return
        html = """
        <html>
        <head><meta charset="UTF-8"><title>SFT低质量样本聚类报告</title></head>
        <body>
        <h1>低质量样本聚类分析报告</h1>
        """
        for i in range(df_clustered['cluster'].nunique()):
            cluster_data = df_clustered[df_clustered['cluster'] == i]
            html += f"<h2>聚类{i} (数量{len(cluster_data)})</h2>"
            html += "<table border='1'><tr><th>指令</th><th>总分</th><th>完整性</th><th>指令遵循</th><th>流畅性</th></tr>"
            for _, row in cluster_data.head(10).iterrows():
                html += f"<tr><td>{row['instruction'][:100]}</td><td>{row['total_score']}</td><td>{row['completeness']}</td><td>{row['instruction_following']}</td><td>{row['fluency']}</td></tr>"
            html += "</table><br>"
        html += "</body></html>"
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html)
        logger.info(f"聚类报告已保存至 {output_path}")


class SFTQualityPipeline:
    def __init__(self, config: Optional[Config] = None):
        self.config = config or Config()
        self.scorer = QualityScorer(self.config)
        self.deduplicator = Deduplicator(self.config)
        self.clusterer = BadcaseClusterer(self.config)

    def load_real_data_from_dir(self, data_dir: str, n_samples: Optional[int] = None) -> pd.DataFrame:
        """递归加载目录下所有 JSON/JSONL 文件，自动适配字段名"""
        files = glob.glob(os.path.join(data_dir, "**", "*.json"), recursive=True) + \
                glob.glob(os.path.join(data_dir, "**", "*.jsonl"), recursive=True)
        if not files:
            raise FileNotFoundError(f"在 {data_dir} 下未找到 .json 或 .jsonl 文件")

        logger.info(f"找到 {len(files)} 个数据文件，开始加载...")
        data = []
        for file_path in files:
            try:
                if file_path.endswith(".jsonl"):
                    with open(file_path, 'r', encoding='utf-8') as f:
                        for line in f:
                            line = line.strip()
                            if not line:
                                continue
                            item = json.loads(line)
                            data.append(item)
                else:  # .json
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = json.load(f)
                        if isinstance(content, list):
                            data.extend(content)
                        else:
                            data.append(content)
            except Exception as e:
                logger.warning(f"读取文件 {file_path} 出错: {e}")

        def map_field(item: dict, candidates: List[str]) -> Any:
            for cand in candidates:
                if cand in item:
                    return item[cand]
            return ""

        mapped_data = []
        for item in data:
            instruction = map_field(item, ["instruction", "question", "prompt", "query", "text"])
            input_text = map_field(item, ["input", "context"])
            output = map_field(item, ["output", "response", "answer", "completion", "target"])
            task_type = map_field(item, ["task_type", "task", "type"])
            if not output:
                continue
            mapped_data.append({
                "instruction": instruction,
                "input": input_text,
                "output": output,
                "task_type": task_type if task_type else "未知"
            })

        df = pd.DataFrame(mapped_data)
        if n_samples and n_samples < len(df):
            df = df.sample(n=n_samples, random_state=self.config.random_seed)
            logger.info(f"随机采样 {n_samples} 条数据")
        logger.info(f"从真实数据加载了 {len(df)} 条有效样本")
        return df

    def _print_verbose_summary(self, df_scored: pd.DataFrame):
        print("\n" + "="*60)
        print("详细统计信息 (verbose=True)")
        print("="*60)

        print("\n=== 各任务类型平均分 ===")
        if 'task_type' in df_scored.columns:
            type_means = df_scored.groupby('task_type')['total_score'].mean().sort_values()
            for task, score in type_means.items():
                print(f"  {task}: {score:.2f}")
        else:
            print("  无 task_type 字段")

        low = df_scored[df_scored['total_score'] < self.config.low_score_threshold]
        print(f"\n=== 低质量样本（总分<{self.config.low_score_threshold}）===")
        print(f"数量: {len(low)} ({len(low)/len(df_scored)*100:.1f}%)")
        if len(low) >= 3:
            clustered = self.clusterer.cluster(df_scored, verbose=False)
            if clustered is not None:
                for i in range(clustered['cluster'].nunique()):
                    cluster_data = clustered[clustered['cluster'] == i]
                    print(f"\n聚类{i} (数量{len(cluster_data)}):")
                    if len(cluster_data) > 0:
                        typical = cluster_data['instruction'].iloc[0]
                        print(f"  典型指令: {typical[:80]}")
                        avg_score = cluster_data['total_score'].mean()
                        low_comp = (cluster_data['completeness'] < 2).mean()
                        low_follow = (cluster_data['instruction_following'] < 2).mean()
                        low_flu = (cluster_data['fluency'] < 2).mean()
                        print(f"  平均质量分: {avg_score:.1f}")
                        print(f"  主要问题: 完整性不足{low_comp:.0%} 指令遵循不足{low_follow:.0%} 流畅性差{low_flu:.0%}")

        high = df_scored[df_scored['total_score'] >= self.config.high_score_threshold]
        print(f"\n=== 高分样本示例（前3条）===")
        for idx, row in high.head(3).iterrows():
            print(f"\n指令: {row['instruction'][:60]}...")
            print(f"输出: {row['output'][:100]}...")
            print(f"总分: {row['total_score']} (完整性:{row['completeness']}, 遵循:{row['instruction_following']}, 流畅:{row['fluency']})")

        print("\n=== 分数分布 ===")
        print(df_scored['total_score'].value_counts().sort_index())

        plt.figure(figsize=(8,4))
        plt.hist(df_scored['total_score'], bins=range(0, 14), color='skyblue', edgecolor='black')
        plt.xlabel('质量总分 (0-12)')
        plt.ylabel('样本数')
        plt.title('SFT数据质量分布')
        plt.show()
        print("="*60)

    def run(self, data_dir: str, verbose: bool = False, n_samples: Optional[int] = None) -> Dict[str, Any]:
        """
        运行流水线（仅使用真实数据）
        :param data_dir: 真实数据集目录（必填）
        :param verbose: 是否打印详细统计信息
        :param n_samples: 采样数量，覆盖 config 中的设置
        """
        logger.info("=== SFT质量评估流水线启动 ===")

        if n_samples is not None:
            self.config.n_samples = n_samples

        if not os.path.exists(data_dir):
            raise FileNotFoundError(f"数据目录不存在: {data_dir}")

        df_raw = self.load_real_data_from_dir(data_dir, self.config.n_samples)

        if df_raw.empty:
            raise ValueError("加载的数据为空，请检查数据格式")

        # 去重
        if self.config.exact_dedup:
            df_raw = self.deduplicator.exact_deduplication(df_raw)
        if self.config.near_dedup:
            df_raw = self.deduplicator.near_deduplication(df_raw)

        # 评分
        df_scored = self.scorer.evaluate_dataset(df_raw)

        # 低质量聚类
        low_clusters = self.clusterer.cluster(df_scored, verbose=verbose)

        # 输出
        os.makedirs(self.config.output_dir, exist_ok=True)
        high = df_scored[df_scored['total_score'] >= self.config.high_score_threshold].copy()
        low = df_scored[df_scored['total_score'] < self.config.low_score_threshold].copy()

        logger.info(f"高质量数据数量: {len(high)} / {len(df_scored)} ({len(high)/len(df_scored)*100:.1f}%)")
        if len(high) > 0:
            high_path = os.path.join(self.config.output_dir, self.config.high_quality_file)
            high[['instruction', 'input', 'output', 'task_type', 'total_score']].to_json(
                high_path, orient='records', force_ascii=False, indent=2
            )
            logger.info(f"高质量数据已保存至 {high_path}")
        if len(low) > 0:
            low_path = os.path.join(self.config.output_dir, self.config.low_quality_file)
            low[['instruction', 'output', 'total_score']].to_csv(low_path, index=False, encoding='utf-8-sig')
            logger.info(f"低质量数据已保存至 {low_path}")

        if low_clusters is not None:
            report_path = os.path.join(self.config.output_dir, self.config.cluster_report_file)
            self.clusterer.generate_cluster_report(low_clusters, report_path)

        if verbose:
            self._print_verbose_summary(df_scored)

        logger.info("=== 流水线执行完毕 ===")
        return {
            "raw_data": df_raw,
            "scored_data": df_scored,
            "low_clusters": low_clusters,
            "high_quality": high
        }


# ==================== 使用示例 ====================
if __name__ == "__main__":
    # 示例1：纯规则评分（默认）
    # config = Config(
    #     exact_dedup=True,
    #     near_dedup=True,
    #     high_score_threshold=8,
    #     low_score_threshold=6,
    #     output_dir="output_sft",
    #     n_samples=500,
    #     enable_generic_following=True
    # )

    # 示例2：启用LLM混合评分（需设置环境变量 DEEPSEEK_API_KEY）
    config = Config(
        exact_dedup=True,
        near_dedup=True,
        high_score_threshold=8,
        low_score_threshold=6,
        output_dir="output_sft",
        n_samples=None,
        enable_generic_following=True,
        # 新增LLM配置
        enable_llm_scoring=True,          # 开启混合评分
        llm_sample_ratio=0.2,             # 20%边界样本调用LLM
        llm_score_lower=5,
        llm_score_upper=8,
        llm_rule_weight=0.4,
        llm_model_weight=0.6,
        # llm_api_key="your-key-here"     # 也可直接传入，建议用环境变量
    )

    pipeline = SFTQualityPipeline(config)

    # 真实数据集路径（请修改为你的实际路径）
    data_path = r"./SFT数据集"

    # 运行，采样500条以便快速验证
    result = pipeline.run(data_dir=data_path, verbose=True, n_samples=200)

    print(f"\n处理完成！高质量样本数: {len(result['high_quality'])}")